# 03b. Despliegue del Modelo e Integración RAG Vectorial (Fase de Validación)

## Objetivos de este Notebook
1.  **Carga Dual:** Desplegar el LLM (**Phi-3-mini-4k-instruct**) y el modelo de Embeddings (**BAAI/bge-m3**) en la GPU.
2.  **Conexión Distribuida:** Establecer comunicación con Elasticsearch (alojado en CPU/Valencia) mediante túnel SSH.
3.  **Pipeline RAG Semántico:** Implementar la función de búsqueda vectorial (kNN) con configuración estricta (**Top-k=1**, **Match Perfecto**).
4.  **Validación Técnica:** Realizar las mismas pruebas de cordura que en el BM25 para comparar resultados.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from sentence_transformers import SentenceTransformer

# 1. Configuración de Hardware
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Hardware detectado: {torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'}")

# 2. Carga del Modelo de Embeddings (NUEVO PARA FASE VECTORIAL)
print("Cargando modelo de Embeddings: BAAI/bge-m3...")
embed_model = SentenceTransformer('BAAI/bge-m3', device=device)

# 3. Selección y Carga del LLM 
MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"
print(f"Cargando modelo LLM: {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=False)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",             
    torch_dtype=torch.float16    
)

# 4. Crear Pipeline de Generación
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

print("\n ¡Modelos cargados y listos para inferencia!")

/home/javierruiz/miniconda3/envs/environment/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Hardware detectado: NVIDIA GeForce RTX 4090
Cargando modelo de Embeddings: BAAI/bge-m3...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3094.42it/s, Materializing param=pooler.dense.weight]                               


Cargando modelo LLM: microsoft/Phi-3-mini-4k-instruct...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 195/195 [00:01<00:00, 124.81it/s, Materializing param=model.norm.weight]                              



 ¡Modelos cargados y listos para inferencia!


In [2]:
# 5. Prueba de Inferencia (Sin RAG todavía)
messages = [
    {"role": "user", "content": "¿Podrías explicarme brevemente qué es la inflación económica?"},
]

generation_args = {
    "max_new_tokens": 150,     
    "return_full_text": False, 
    "temperature": 0.1,        
    "do_sample": True,
}

print(" Generando respuesta...")
output = pipe(messages, **generation_args)
print("\n" + "="*50)
print(output[0]['generated_text'][-1]['content'] if isinstance(output[0]['generated_text'], list) else output[0]['generated_text'])
print("="*50)

Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Generando respuesta...

 La inflación económica es un fenómeno que se refiere al aumento generalizado y sostenido de los precios de bienes y servicios en una economía. Este proceso reduce el poder adquisitivo de la moneda, lo que significa que con la misma cantidad de dinero, los consumidores pueden comprar menos de lo que podían antes. La inflación puede ser causada por diversos factores, como un aumento en la oferta de dinero, un aumento en los costos de producción, o una disminución en la demanda de bienes y servicios. A largo plazo, una inflación moderada es aceptada por


In [3]:
from elasticsearch import Elasticsearch

# Conexión a Elasticsearch
es = Elasticsearch("http://127.0.0.1:9250")

try:
    if es.ping():
        print("PING EXITOSO")
        print(f" Conectado a: {es.info()['name']}")
    else:
        print("El servidor no responde. ¿Está el túnel abierto?")
except Exception as e:
    print(f"Error inesperado: {e}")

PING EXITOSO
 Conectado a: valencia


In [35]:
def ask_rag_vectorial(query, top_k=1): 
    """
    Función RAG Vectorial: Recupera 1 noticia (kNN) y genera respuesta basada estrictamente en ella.
    Retorna un diccionario con los datos separados.
    """
    # --- A. BÚSQUEDA VECTORIAL (kNN) ---
    vector_pregunta = embed_model.encode(query).tolist()
    
    query_knn = {
        "field": "vector_texto",
        "query_vector": vector_pregunta,
        "k": top_k,
        "num_candidates": 50
    }
    
    try:
        response = es.search(
            index="noticias_tfg_vectores", 
            knn=query_knn, 
            _source=["title", "body", "date", "source"], 
            size=top_k
        )
        hits = response['hits']['hits']
    except Exception as e:
        return {"error": f"Error en Elasticsearch: {e}"}
    
    if not hits:
        return {"error": " NO ENCONTRADO: No hay ninguna noticia cercana a esta búsqueda."}

    # --- B. EXTRACCIÓN ---
    contexto_unico = ""
    for i, hit in enumerate(hits):
        doc = hit['_source']
        contexto_unico += f"""
        --- NOTICIA {i+1} ---
        TITULO: {doc.get('title')}
        FECHA: {doc.get('date', 'Desconocida')}
        FUENTE: {doc.get('source', 'Desconocida')}
        CONTENIDO: {doc.get('body')}
        \n"""

    # --- C. PROMPT ---
    messages = [
        {"role": "user", "content": f"""
        Eres un sistema de verificación de datos (Fact-Checking). 
        Tu objetivo es responder a la pregunta usando ÚNICAMENTE el texto que te proporciono abajo.

        REGLAS CRÍTICAS:
        1. Si la respuesta no aparece en el texto, responde exactamente: "No tengo información suficiente en mis archivos".
        2. Formula tu respuesta usando solo los datos del texto. Puedes conectar ideas que estén en la noticia, pero NUNCA uses conocimiento externo ni inventes datos.
        3. No menciones otras noticias que no sean las proporcionadas.
        4. Se claro y conciso.

        ### TEXTO DE REFERENCIA:
        {contexto_unico}
        
        ### PREGUNTA:
        {query}
        """}
    ]

    # --- D. GENERACIÓN ---
    outputs = pipe(
        messages,
        max_new_tokens=256,
        do_sample=False,
        temperature=0.0, 
    )
    
    # --- E. RETORNO ESTRUCTURADO  ---
    respuesta_limpia = outputs[0]['generated_text'][-1]['content']
    
    return {
        "titulo": [hit['_source'].get('title') for hit in hits],
        "contenido": [hit['_source'].get('body') for hit in hits],
        "fecha": [hit['_source'].get('date', 'Desconocida') for hit in hits],
        "fuente": [hit['_source'].get('source', 'Desconocida') for hit in hits],
        "respuesta_rag": respuesta_limpia
    }


In [34]:
pregunta_tfg = "¿Qué material se utilizó específicamente para las palas de ping pong de los deportistas españoles en los Juegos de París 2024?"
print(f" PREGUNTA: {pregunta_tfg}\n")

print("--- RESPUESTA SLM (SIN RAG) ---")
res_base = pipe([{"role": "user", "content": pregunta_tfg}], max_new_tokens=150)
print(res_base[0]['generated_text'][-1]['content']) 

print("\n")

datos = ask_rag_vectorial(pregunta_tfg, top_k=2)
print("--- RESPUESTA SISTEMA RAG VECTORIAL ---")
if "error" in datos:
    print(datos["error"])
else:
    print(f"NOTICIA: {datos['titulo']}")
    print(f"TEXTO: {datos['contenido'][:300]} (...)")
    print(f"RESPUESTA: {datos['respuesta_rag']}")

 PREGUNTA: ¿Qué material se utilizó específicamente para las palas de ping pong de los deportistas españoles en los Juegos de París 2024?

--- RESPUESTA SLM (SIN RAG) ---
 No hay información disponible sobre los materiales específicos de las palas de ping pong utilizadas por los deportistas españoles en los Juegos de París 2024, ya que estos eventos aún no han tenido lugar. Sin embargo, es conocido que las palas de ping pong generalmente están hechas de plástico de baja densidad, una combinación de celuloide y resina. Las palas también pueden estar recubiertas con materiales como aluminio para mejorar su durabilidad y rebound.


--- RESPUESTA SISTEMA RAG VECTORIAL ---
NOTICIA: ['¿Cuántas medallas de oro, plata y bronce lleva España en París y cómo va hoy jueves el medallero de los Jue...', 'España en los Juegos Olímpicos de Paris 2024: todos los deportistas clasificados']
TEXTO: ['Los Juegos Olímpicos de París 2024 entran en la recta final. Desde el miércoles 24 de julio hasta el domin

In [36]:
import os
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import warnings
import transformers

# 0. Modo silencioso para evitar los avisos rojos molestos
transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore")

# 1. Configurar el Juez (Llama 3 8B vía Groq)
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    print("ADVERTENCIA: No se encontró GROQ_API_KEY en el entorno .env.")

# 1. Configurar el Juez (Llama 3.1 8B vía Groq)
llm_juez = ChatGroq(
    model="llama-3.1-8b-instant",  # <--- ¡ESTE ES EL CAMBIO!
    temperature=0, 
    api_key=GROQ_API_KEY
)

prompt_evaluacion = ChatPromptTemplate.from_messages([
    ("system", """Eres un evaluador comprensivo de sistemas RAG. 
    Tu tarea es comparar la 'Respuesta Esperada' con la 'Respuesta Generada'.
    
    Regla fundamental: Si la Respuesta Generada contiene la idea principal o la solución básica de la Respuesta Esperada, debes darla por válida, aunque falten detalles menores o la redacción sea muy distinta. 
    
    Responde ÚNICAMENTE con la palabra "CORRECTO" si la esencia es la misma, o "INCORRECTO" si la IA dice que no tiene información, si se contradice o si la idea principal es totalmente errónea."""),
    ("human", "Respuesta Esperada:\n{esperada}\n\nRespuesta Generada por el RAG:\n{generada}")
])

juez_chain = prompt_evaluacion | llm_juez | StrOutputParser()

# 2. Cargar el CSV con la batería de preguntas
df_preguntas = pd.read_csv("bateria_preguntas_tfg.csv")
resultados_vectorial = []

print(f"Iniciando evaluación vectorial masiva para {len(df_preguntas)} preguntas...")
print("Usando RAG Vectorial (top_k=2) y Juez Llama-3-8B...\n")

# 3. Bucle de evaluación
# 3. Bucle de evaluación
for index, row in tqdm(df_preguntas.iterrows(), total=len(df_preguntas)):
    pregunta = row['pregunta']
    esperada = row['respuesta_esperada']
    
    # ¡Cambiado a top_k=2 como querías!
    datos = ask_rag_vectorial(pregunta, top_k=2) 
    resp_generada = datos.get('respuesta_rag', datos.get('error', 'Error desconocido'))
    
    # El juez de Groq evalúa
    evaluacion = juez_chain.invoke({"esperada": esperada, "generada": resp_generada}).strip().upper()
    
    if "INCORRECTO" in evaluacion: 
        evaluacion = "INCORRECTO"
    elif "CORRECTO" in evaluacion: 
        evaluacion = "CORRECTO"
    else:
        evaluacion = "INCORRECTO" # Por si el juez se vuelve loco y responde otra cosa
    
    # Guardamos los resultados
    resultados_vectorial.append({
        "Pregunta": pregunta,
        "Esperada": esperada,
        "Respuesta_Generada": resp_generada,
        "Evaluacion": evaluacion
    })

# 4. Guardar y mostrar resultados
df_res_vect = pd.DataFrame(resultados_vectorial)
df_res_vect.to_csv("resultados_solo_vectorial.csv", index=False)

aciertos = (df_res_vect['Evaluacion'] == 'CORRECTO').sum()
total = len(df_res_vect)
porcentaje = (aciertos / total) * 100 if total > 0 else 0

print(f"\nEvaluación terminada. Resultados guardados en 'resultados_solo_vectorial.csv'.")
print(f"RENDIMIENTO DEL RAG VECTORIAL: {aciertos}/{total} Aciertos ({porcentaje:.1f}%)")

# Mostramos las primeras 5 filas para un vistazo rápido
display(df_res_vect[['Pregunta', 'Respuesta_Generada', 'Evaluacion']].head())

Iniciando evaluación vectorial masiva para 80 preguntas...
Usando RAG Vectorial (top_k=2) y Juez Llama-3-8B...



  0%|          | 0/80 [00:00<?, ?it/s]

100%|██████████| 80/80 [03:05<00:00,  2.32s/it]


Evaluación terminada. Resultados guardados en 'resultados_solo_vectorial.csv'.
RENDIMIENTO DEL RAG VECTORIAL: 69/80 Aciertos (86.2%)


,Pregunta,Respuesta_Generada,Evaluacion
0,¿A cuanto llego a subir Trump los aranceles a ...,Los aranceles a China llegan a subir hasta el...,CORRECTO
1,¿Cuánto dinero perdió Elon Musk tras el anunci...,Según los datos proporcionados en la Noticia ...,CORRECTO
2,¿Qué opinó el Rey de España sobre los arancele...,No hay información en el texto de referencia ...,CORRECTO
3,¿Qué declaraciones hizo Elon Musk sobre los ar...,No tengo información suficiente en mis archivos.,CORRECTO
4,¿Qué relación quiere Elon Musk entre Estados U...,Según las declaraciones de Elon Musk en Itali...,CORRECTO


In [37]:
# Guardamos el csv
df_res_vect.to_csv("resultados_vectores_tfg.csv", index=False)